In [51]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1) Take a sample of 1000 from ChEMBL.

In [52]:
ddir = '/home/kat/Repos/SALSA/data/'
df = pd.read_csv(os.path.join(ddir,'chembl_valid_lte_120.csv'))
display(df)

seedy = 666
size = 1000

df_samp = df.sample(size, random_state=seedy)
df_samp = df_samp.reset_index(drop=True)
display(df_samp)

,smiles,len
0,CCO,3
1,C,1
2,CO,2
3,NCCS,4
4,NCCN,4
...,...,...
1476101,CC1C2CCC3C4CCC5Cc6nc7c(nc6CC5(C)C4CC(=O)C32COC...,120
1476102,CNC(=O)c1cc(Oc2ccc(Nc3ccc(Cl)c(C(F)(F)F)c3)cc2...,51
1476103,Cc1cc(C)c2c(c1)N(CCCN1CCN(c3ccccc3)CC1)C(=O)c1...,58
1476104,CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC(c5ccc(F)cc5F...,81


,smiles,len
0,COc1c(Br)cc2c(c1Br)OC(=O)C2=C(C)C,33
1,COc1cc(CC(C)(O)Cc2ccccc2F)nc(OC)n1,34
2,C=C(C)CSc1nnc2n(C)c(=O)c3c4c(sc3n12)CCC4,40
3,Cc1ncccc1C(=O)NC1CC(=O)N(CC(F)(F)F)C1,37
4,Cc1ccc(S(=O)(=O)NCc2cccc([N+](=O)[O-])c2)cc1,44
...,...,...
995,CCC1SC2=NC(C)=C(C(=O)OCCOC)C(C=Cc3ccccc3OC)N2C1=O,49
996,OC(CNCc1ccc(F)cc1)COc1cccc2[nH]ccc12,36
997,O=S(=O)(c1ccccc1)c1ccc(-c2nn(CN3CCOCC3)c(=S)n2...,60
998,Cc1ccc(CNC(=O)NC2CCCN(c3ncccn3)C2)cn1,37


## 2) Calculate all properties.

In [48]:
from tqdm import tqdm
from property_predictors import surface_predictor

all_props = []

for sm in tqdm(df_samp.smiles.values,total=len(df_samp)):
    props = surface_predictor(sm)
    all_props.append(props)

100%|██████████| 1000/1000 [00:02<00:00, 477.55it/s]


## 3) Get Mahalanobis distance. (per mol) 

In [49]:
import numpy as np
from scipy.spatial import distance

df_samp['mahalanobis'] = -1

props_arr = np.vstack([np.array(x) for x in all_props])
print(props_arr.shape)

# Get rid of all-zero properties (to ensure det!=0)
to_keep = []
for i in range(props_arr.shape[1]):
    s = sum(props_arr[:,i])
    if s==0:
        continue
    else:
        to_keep.append(i)

props_arr_clean = props_arr[:,to_keep]
print(props_arr_clean.shape)

(1000, 49)
(1000, 47)


In [50]:
dists = []

for i,row in df_samp.iterrows():
    
    samp_arr = np.expand_dims(props_arr_clean[i],axis=0)    
    
    d = distance.cdist(samp_arr, props_arr_clean, 'mahalanobis')
    dists.append(d)
    
    print(len(d[0]),d)
    break

1000 [[ 0.          8.43397538  8.97466726  8.02676529  8.68656343 11.1331645
   8.23384394  8.53574656 10.03382568 10.34025052  8.75111246  6.58281462
   8.41165117  9.568522    7.4355145   6.09712094  6.95625478  8.10794294
   8.50215669  8.79561774  8.0017659   7.82813554  8.96861136  8.7965605
   7.63629923  8.28968225  9.23663311  8.23056334  9.82000796  8.65067107
   8.38359106  7.80846345 10.46313534  7.75230228  8.23922687  8.28697881
   9.16071179  8.33867009  7.45308934 13.96109807  9.76023777  8.7491159
   9.51867479  9.154405    8.10277378  8.24099893  8.0567046  15.86818077
   8.55560337  7.80892334  8.23617558 14.58550313  8.36477323  9.74044648
   9.93006543  8.03478692  8.05826068  8.27563995  9.38210879  8.39602116
   8.77738823  8.4795686  10.14218299  8.60360793  8.66086511  9.69780164
   8.22388562  8.96232157  9.58184546  8.19382225 12.59781666  7.36484767
   7.4704987   7.86327703  8.4486345   8.79580241  7.20997322 10.50241307
   8.26516492  7.33635074  8.1763531